In [ ]:
# Pretrained Baseline

#This notebook runs a factory-default `yolov8n.pt` model on a held-out test split from `TrafficProject/train`, then saves failure visualizations and a small detection score summary.


In [8]:
from collections import defaultdict
from pathlib import Path
from statistics import mean
import csv
import importlib
import json
import random
import struct
import subprocess
import sys
import zlib


def ensure_packages(packages):
    for package, import_name in packages:
        try:
            importlib.import_module(import_name)
        except ModuleNotFoundError:
            print(f'Installing missing package: {package}')
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', package])


ensure_packages([
    ('numpy', 'numpy'),
    ('ultralytics', 'ultralytics'),
])

import numpy as np
from ultralytics import YOLO

ROOT = Path('/Users/maksymchekhun/Documents/Documents - Maksym’s MacBook Air/Howest/SecondSemester/AdvancedAI/PROJECT/Data/TrafficProject')
ANN_PATH = ROOT / 'train' / '_annotations.coco.json'
IMG_DIR = ROOT / 'train'
OUTPUT_DIR = ROOT / 'baseline_before'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FAILURE_DIR = OUTPUT_DIR / 'failure_screenshots'
FAILURE_DIR.mkdir(parents=True, exist_ok=True)

TEST_SPLIT = 0.15
MAX_TEST_IMAGES = 64
IOU_THRESHOLD = 0.5
RANDOM_SEED = 42

random.seed(RANDOM_SEED)

with ANN_PATH.open('r') as f:
    coco = json.load(f)

categories = {category['id']: category['name'] for category in coco['categories']}
images = {image['id']: image for image in coco['images']}
annotations_by_image = defaultdict(list)
for annotation in coco['annotations']:
    annotations_by_image[annotation['image_id']].append(annotation)

# Map the custom dataset labels to the closest COCO classes that YOLOv8n knows.
# This keeps the baseline honest while still letting us score matches.
label_map = {
    'Bus_Truck': {'bus', 'truck'},
    'Cars': {'car'},
    'Pedestrian': {'person'},
    'Two_Wheeler': {'bicycle', 'motorcycle'},
}

held_out_ids = list(images.keys())
random.shuffle(held_out_ids)
held_out_count = max(1, int(len(held_out_ids) * TEST_SPLIT))
test_image_ids = held_out_ids[: min(held_out_count, MAX_TEST_IMAGES)]


def xywh_to_xyxy(box):
    x, y, w, h = box
    x = float(x)
    y = float(y)
    w = float(w)
    h = float(h)
    return [x, y, x + w, y + h]


def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    inter_w = max(0.0, inter_x2 - inter_x1)
    inter_h = max(0.0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter_area
    return 0.0 if union <= 0 else inter_area / union


def _draw_rectangle(image, box, color, thickness=3):
    x1, y1, x2, y2 = [int(round(value)) for value in box]
    x1 = max(0, min(image.shape[1] - 1, x1))
    x2 = max(0, min(image.shape[1] - 1, x2))
    y1 = max(0, min(image.shape[0] - 1, y1))
    y2 = max(0, min(image.shape[0] - 1, y2))
    if x2 <= x1 or y2 <= y1:
        return
    for offset in range(thickness):
        top = min(image.shape[0] - 1, y1 + offset)
        bottom = max(0, y2 - offset)
        left = min(image.shape[1] - 1, x1 + offset)
        right = max(0, x2 - offset)
        image[top, x1:x2 + 1] = color
        image[bottom, x1:x2 + 1] = color
        image[y1:y2 + 1, left] = color
        image[y1:y2 + 1, right] = color


def _pad_border(image, color, thickness=2):
    image[:thickness, :, :] = color
    image[-thickness:, :, :] = color
    image[:, :thickness, :] = color
    image[:, -thickness:, :] = color


def draw_boxes(image, ground_truth, predictions):
    canvas = image.copy()
    _pad_border(canvas, (255, 255, 255), thickness=2)
    for item in ground_truth:
        _draw_rectangle(canvas, item['box'], (0, 200, 0), thickness=3)
    for item in predictions:
        _draw_rectangle(canvas, item['box'], (220, 0, 0), thickness=3)
    return canvas


def combine_before_after(image, ground_truth, predictions):
    gt_image = draw_boxes(image, ground_truth, [])
    pred_image = draw_boxes(image, [], predictions)
    return np.concatenate([gt_image, pred_image], axis=1)


def write_png(image, output_path):
    if image.dtype != np.uint8:
        image = np.clip(image, 0, 255).astype(np.uint8)
    if image.ndim == 2:
        image = np.repeat(image[:, :, None], 3, axis=2)
    if image.shape[2] > 3:
        image = image[:, :, :3]
    height, width, _ = image.shape
    raw = b''.join(b'\x00' + image[row].tobytes() for row in range(height))

    def chunk(tag, data):
        return struct.pack('!I', len(data)) + tag + data + struct.pack('!I', zlib.crc32(tag + data) & 0xFFFFFFFF)

    png = b''.join([
        b'\x89PNG\r\n\x1a\n',
        chunk(b'IHDR', struct.pack('!2I5B', width, height, 8, 2, 0, 0, 0)),
        chunk(b'IDAT', zlib.compress(raw, 9)),
        chunk(b'IEND', b''),
    ])
    output_path.write_bytes(png)


model = YOLO('yolov8n.pt')

per_image_rows = []
all_detections = []

for index, image_id in enumerate(test_image_ids, start=1):
    image_info = images[image_id]
    image_path = IMG_DIR / image_info['file_name']

    gt_items = []
    for annotation in annotations_by_image[image_id]:
        category_name = categories[annotation['category_id']]
        if category_name not in label_map:
            continue
        gt_items.append({
            'label': category_name,
            'box': xywh_to_xyxy(annotation['bbox']),
        })

    result = model.predict(source=str(image_path), imgsz=640, conf=0.25, verbose=False)[0]
    original = result.orig_img[..., ::-1].copy()
    predictions = []
    if result.boxes is not None and len(result.boxes) > 0:
        for box, cls_id, conf in zip(result.boxes.xyxy.tolist(), result.boxes.cls.tolist(), result.boxes.conf.tolist()):
            label = model.names[int(cls_id)]
            predictions.append({'label': label, 'box': box, 'conf': float(conf)})

    used_predictions = set()
    true_positives = 0
    false_negatives = 0
    for gt in gt_items:
        allowed = label_map.get(gt['label'], set())
        best_match = None
        best_iou = 0.0
        for pred_index, prediction in enumerate(predictions):
            if pred_index in used_predictions:
                continue
            if prediction['label'] not in allowed:
                continue
            iou = box_iou(gt['box'], prediction['box'])
            if iou >= IOU_THRESHOLD and iou > best_iou:
                best_iou = iou
                best_match = pred_index
        if best_match is None:
            false_negatives += 1
        else:
            used_predictions.add(best_match)
            true_positives += 1

    false_positives = len(predictions) - len(used_predictions)
    recall = true_positives / max(1, len(gt_items))
    precision = true_positives / max(1, true_positives + false_positives)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

    per_image_rows.append({
        'image_id': image_id,
        'file_name': image_info['file_name'],
        'ground_truth_boxes': len(gt_items),
        'predicted_boxes': len(predictions),
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_negatives': false_negatives,
        'precision': precision,
        'recall': recall,
        'f1': f1,
    })

    all_detections.append({
        'image_id': image_id,
        'file_name': image_info['file_name'],
        'ground_truth_boxes': len(gt_items),
        'predicted_boxes': len(predictions),
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_negatives': false_negatives,
        'recall': recall,
        'precision': precision,
        'f1': f1,
        'ground_truth': gt_items,
        'predictions': predictions,
        'original': original,
    })

results_rows = sorted(per_image_rows, key=lambda row: (row['recall'], -row['ground_truth_boxes']))
summary = {
    'model': 'yolov8n.pt',
    'test_images': len(results_rows),
    'test_split': TEST_SPLIT,
    'iou_threshold': IOU_THRESHOLD,
    'mean_precision': mean(row['precision'] for row in results_rows),
    'mean_recall': mean(row['recall'] for row in results_rows),
    'mean_f1': mean(row['f1'] for row in results_rows),
    'micro_precision': sum(row['true_positives'] for row in results_rows) / max(1, sum(row['true_positives'] for row in results_rows) + sum(row['false_positives'] for row in results_rows)),
    'micro_recall': sum(row['true_positives'] for row in results_rows) / max(1, sum(row['true_positives'] for row in results_rows) + sum(row['false_negatives'] for row in results_rows)),
}
summary['micro_f1'] = 0.0 if summary['micro_precision'] + summary['micro_recall'] == 0 else 2 * summary['micro_precision'] * summary['micro_recall'] / (summary['micro_precision'] + summary['micro_recall'])

with (OUTPUT_DIR / 'baseline_per_image_metrics.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(results_rows[0].keys()))
    writer.writeheader()
    writer.writerows(results_rows)

with (OUTPUT_DIR / 'baseline_summary.json').open('w') as f:
    json.dump(summary, f, indent=2)

print('Baseline summary')
for key, value in summary.items():
    if key == 'model':
        print(f'{key}: {value}')
    else:
        print(f'{key}: {value:.4f}' if isinstance(value, float) else f'{key}: {value}')

print('\nWorst images by recall:')
worst_rows = results_rows[:5]
for row in worst_rows:
    print(f"{row['file_name']}: recall={row['recall']:.3f}, precision={row['precision']:.3f}, gt={int(row['ground_truth_boxes'])}, pred={int(row['predicted_boxes'])}")

worst_image_ids = [row['image_id'] for row in worst_rows]
for rank, image_id in enumerate(worst_image_ids, start=1):
    record = next(item for item in all_detections if item['image_id'] == image_id)
    canvas = combine_before_after(record['original'], record['ground_truth'], record['predictions'])
    output_path = FAILURE_DIR / f'{rank:02d}_{Path(record["file_name"]).stem}_failure.png'
    write_png(canvas, output_path)
    print(f'saved {output_path}')

print('\nSaved failure screenshots in:', FAILURE_DIR)


Baseline summary
model: yolov8n.pt
test_images: 64
test_split: 0.1500
iou_threshold: 0.5000
mean_precision: 0.0601
mean_recall: 0.0031
mean_f1: 0.0058
micro_precision: 0.0667
micro_recall: 0.0039
micro_f1: 0.0073

Worst images by recall:
frame_0381_y440_x512_jpg.rf.1BVUTH6ZNcWGWqDIh5WH.jpg: recall=0.000, precision=0.000, gt=139, pred=8
frame_0502_y440_x512_jpg.rf.1MS9wmEpArVHSZJgBATE.jpg: recall=0.000, precision=0.000, gt=126, pred=3
frame_0445_y0_x0_jpg.rf.0L98Rcl1oj4jvLQiODCa.jpg: recall=0.000, precision=0.000, gt=109, pred=5
frame_0438_y0_x1024_jpg.rf.34dAEWArqGBtrsohNlrj.jpg: recall=0.000, precision=0.000, gt=102, pred=4
frame_0321_y440_x0_jpg.rf.MEJ11Re2Eb4SoZSJJu5h.jpg: recall=0.000, precision=0.000, gt=80, pred=0
saved /Users/maksymchekhun/Documents/Documents - Maksym’s MacBook Air/Howest/SecondSemester/AdvancedAI/PROJECT/Data/TrafficProject/baseline_before/failure_screenshots/01_frame_0381_y440_x512_jpg.rf.1BVUTH6ZNcWGWqDIh5WH_failure.png
saved /Users/maksymchekhun/Documents/Do

In [10]:
# Confusion matrix for baseline predictions (run Cell 2 first).
if 'all_detections' not in globals() or 'OUTPUT_DIR' not in globals() or 'box_iou' not in globals():
    print('Please run Cell 2 first to compute detections before building the confusion matrix.')
else:
    class_names = ['Bus_Truck', 'Cars', 'Pedestrian', 'Two_Wheeler']
    background = 'background'
    labels = class_names + [background]
    label_to_idx = {label: index for index, label in enumerate(labels)}

    # Map YOLO COCO labels back to your dataset labels.
    pred_to_dataset = {
        'bus': 'Bus_Truck',
        'truck': 'Bus_Truck',
        'car': 'Cars',
        'person': 'Pedestrian',
        'bicycle': 'Two_Wheeler',
        'motorcycle': 'Two_Wheeler',
    }

    matrix = np.zeros((len(labels), len(labels)), dtype=np.int64)

    for record in all_detections:
        gts = record['ground_truth']
        preds = record['predictions']
        used_pred_indices = set()

        # Match each GT to the best unused prediction by IoU.
        for gt in gts:
            gt_label = gt['label']
            best_pred_index = None
            best_iou = 0.0

            for pred_index, pred in enumerate(preds):
                if pred_index in used_pred_indices:
                    continue
                iou = box_iou(gt['box'], pred['box'])
                if iou > best_iou:
                    best_iou = iou
                    best_pred_index = pred_index

            true_idx = label_to_idx[gt_label]
            if best_pred_index is not None and best_iou >= IOU_THRESHOLD:
                used_pred_indices.add(best_pred_index)
                pred_label = pred_to_dataset.get(preds[best_pred_index]['label'], background)
                pred_idx = label_to_idx[pred_label]
                matrix[true_idx, pred_idx] += 1
            else:
                # Missed GT object.
                matrix[true_idx, label_to_idx[background]] += 1

        # Unmatched predictions are false positives on background.
        for pred_index, pred in enumerate(preds):
            if pred_index in used_pred_indices:
                continue
            pred_label = pred_to_dataset.get(pred['label'], background)
            matrix[label_to_idx[background], label_to_idx[pred_label]] += 1

    # Print a readable matrix.
    header = ['true\\pred'] + labels
    col_width = max(max(len(h) for h in header), 12)
    print('Confusion Matrix (counts)')
    print(''.join(text.ljust(col_width) for text in header))
    for row_label, row in zip(labels, matrix):
        print(row_label.ljust(col_width) + ''.join(str(int(value)).ljust(col_width) for value in row))

    # Save matrix as CSV.
    cm_csv_path = OUTPUT_DIR / 'baseline_confusion_matrix.csv'
    with cm_csv_path.open('w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        for row_label, row in zip(labels, matrix):
            writer.writerow([row_label] + [int(value) for value in row])

    # Quick summary numbers.
    diag_sum = int(np.trace(matrix[:-1, :-1]))
    total_gt = int(matrix[:-1, :].sum())
    detection_rate = 0.0 if total_gt == 0 else diag_sum / total_gt

    print(f'\nDiagonal true-class hits (without background): {diag_sum}')
    print(f'Total GT objects: {total_gt}')
    print(f'Class-level hit rate: {detection_rate:.4f}')
    print(f'Saved: {cm_csv_path}')


Confusion Matrix (counts)
true\pred   Bus_Truck   Cars        Pedestrian  Two_Wheeler background  
Bus_Truck   0           0           0           0           45          
Cars        11          10          2           1           1606        
Pedestrian  0           1           1           0           1171        
Two_Wheeler 0           0           0           0           1           
background  1           0           2           0           21          

Diagonal true-class hits (without background): 11
Total GT objects: 2849
Class-level hit rate: 0.0039
Saved: /Users/maksymchekhun/Documents/Documents - Maksym’s MacBook Air/Howest/SecondSemester/AdvancedAI/PROJECT/Data/TrafficProject/baseline_before/baseline_confusion_matrix.csv


In [ ]:
# Fine-Tuned Model (After)

#Train the same `yolov8n.pt` model on 3,200 Roboflow images, then evaluate on the held-out test split.

In [11]:
import os
import shutil

# -------------------------
# Configuration
# -------------------------
TRAIN_IMAGE_COUNT = 3200
EPOCHS = 50  # set to 100 if you want a longer run
BATCH = 16
IMGSZ = 640
RUN_NAME = f'yolov8n_finetuned_topdown_{EPOCHS}e'

SPLIT_ROOT = ROOT / 'split_3200'
IMAGES_TRAIN = SPLIT_ROOT / 'images' / 'train'
IMAGES_TEST = SPLIT_ROOT / 'images' / 'test'
LABELS_TRAIN = SPLIT_ROOT / 'labels' / 'train'
LABELS_TEST = SPLIT_ROOT / 'labels' / 'test'
DATA_YAML = SPLIT_ROOT / 'data.yaml'
RUN_OUTPUT_ROOT = ROOT / 'runs'
PRED_OUTPUT = ROOT / 'finetuned_test_predictions'
PRED_OUTPUT.mkdir(parents=True, exist_ok=True)

for folder in [IMAGES_TRAIN, IMAGES_TEST, LABELS_TRAIN, LABELS_TEST]:
    folder.mkdir(parents=True, exist_ok=True)

# -------------------------
# Build train/test split
# -------------------------
all_ids = list(images.keys())
random.seed(RANDOM_SEED)
random.shuffle(all_ids)

train_ids = all_ids[:TRAIN_IMAGE_COUNT]
test_ids = all_ids[TRAIN_IMAGE_COUNT:]

if len(train_ids) < TRAIN_IMAGE_COUNT:
    raise RuntimeError(f'Not enough images for a 3200-image train split. Found: {len(all_ids)}')
if len(test_ids) == 0:
    raise RuntimeError('No test images left after split.')

train_id_set = set(train_ids)

# Use only the four relevant classes for training.
class_order = ['Bus_Truck', 'Cars', 'Pedestrian', 'Two_Wheeler']
class_to_idx = {name: idx for idx, name in enumerate(class_order)}
category_id_to_name = {category['id']: category['name'] for category in coco['categories']}

# Group annotations per image for faster writing.
anns_by_image = defaultdict(list)
for ann in coco['annotations']:
    anns_by_image[ann['image_id']].append(ann)


def safe_link_or_copy(src, dst):
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except Exception:
        shutil.copy2(src, dst)


def yolo_line_from_bbox(bbox_xywh, image_w, image_h, class_idx):
    x, y, w, h = [float(value) for value in bbox_xywh]
    cx = (x + w / 2.0) / float(image_w)
    cy = (y + h / 2.0) / float(image_h)
    nw = w / float(image_w)
    nh = h / float(image_h)
    return f"{class_idx} {cx:.8f} {cy:.8f} {nw:.8f} {nh:.8f}"


for image_id in all_ids:
    image_info = images[image_id]
    src_img = IMG_DIR / image_info['file_name']

    if image_id in train_id_set:
        dst_img = IMAGES_TRAIN / image_info['file_name']
        dst_lbl = LABELS_TRAIN / f"{Path(image_info['file_name']).stem}.txt"
    else:
        dst_img = IMAGES_TEST / image_info['file_name']
        dst_lbl = LABELS_TEST / f"{Path(image_info['file_name']).stem}.txt"

    safe_link_or_copy(src_img, dst_img)

    yolo_lines = []
    for ann in anns_by_image[image_id]:
        name = category_id_to_name.get(ann['category_id'])
        if name not in class_to_idx:
            continue
        yolo_lines.append(
            yolo_line_from_bbox(
                ann['bbox'],
                image_info['width'],
                image_info['height'],
                class_to_idx[name],
            )
        )

    dst_lbl.write_text('\n'.join(yolo_lines) + ('\n' if yolo_lines else ''))

# Build YOLO dataset yaml.
yaml_text = '\n'.join([
    f"path: {SPLIT_ROOT}",
    "train: images/train",
    "val: images/test",
    f"names: {class_order}",
])
DATA_YAML.write_text(yaml_text + '\n')

print('Dataset split ready:')
print(f'  train images: {len(train_ids)}')
print(f'  test images:  {len(test_ids)}')
print(f'  data yaml:    {DATA_YAML}')

# -------------------------
# Train YOLOv8n on 3200 images
# -------------------------
finetuned_model = YOLO('yolov8n.pt')
train_result = finetuned_model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=str(RUN_OUTPUT_ROOT),
    name=RUN_NAME,
    exist_ok=True,
    verbose=True,
)

best_weights = RUN_OUTPUT_ROOT / RUN_NAME / 'weights' / 'best.pt'
print(f'Best checkpoint: {best_weights}')

# -------------------------
# Evaluate on held-out test split
# -------------------------
trained_model = YOLO(str(best_weights))
val_metrics = trained_model.val(data=str(DATA_YAML), split='val', imgsz=IMGSZ, batch=BATCH)

print('\nValidation metrics on held-out test split:')
print(f"mAP50:    {float(val_metrics.box.map50):.4f}")
print(f"mAP50-95: {float(val_metrics.box.map):.4f}")

# Save a sample of predictions from test split for visual confirmation.
test_sample_paths = [str(IMAGES_TEST / images[i]['file_name']) for i in test_ids[:24]]
trained_model.predict(
    source=test_sample_paths,
    imgsz=IMGSZ,
    conf=0.25,
    save=True,
    project=str(PRED_OUTPUT),
    name=f'pred_{RUN_NAME}',
    exist_ok=True,
    verbose=False,
)

print('\nSaved test predictions to:')
print(PRED_OUTPUT / f'pred_{RUN_NAME}')


Dataset split ready:
  train images: 3200
  test images:  816
  data yaml:    /Users/maksymchekhun/Documents/Documents - Maksym’s MacBook Air/Howest/SecondSemester/AdvancedAI/PROJECT/Data/TrafficProject/split_3200/data.yaml
Ultralytics 8.4.41 🚀 Python-3.14.0 torch-2.11.0 CPU (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/maksymchekhun/Documents/Documents - Maksym’s MacBook Air/Howest/SecondSemester/AdvancedAI/PROJECT/Data/TrafficProject/split_3200/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v

KeyboardInterrupt: 